# Lezione 4 · Vettori, angoli e direzioni

**Intelligenza artificiale · Classe quinta**

Se i dati sono vettori, possiamo confrontarli con la **similarità del coseno**, fare operazioni come *re − uomo + donna* e proiettarli su una **direzione**. Oggi impariamo questi tre strumenti, più un modo per **vedere** spazi con troppe dimensioni.

Riferimenti sul libro: pp. 195–199 (il neurone), box a p. 235 (LLM).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.decomposition import PCA

iris = load_iris()
X = iris.data                              # 150 fiori x 4 misure
specie = iris.target_names[iris.target]    # nome della specie di ogni fiore

## 1. Da dove vengono i numeri delle parole?

Un **neurone** artificiale fa una somma pesata dei suoi ingressi, cioè un **prodotto scalare** tra il vettore degli ingressi e il vettore dei pesi (libro, pp. 195–199):

In [ ]:
def neurone(x, w, b):
    return 1 if np.dot(w, x) + b > 0 else 0

w, b = np.array([-2, -2]), 3          # pesi scelti per realizzare la porta NAND
for x in [(0, 0), (0, 1), (1, 0), (1, 1)]:
    print(x, "->", neurone(np.array(x), w, b))

In una rete con molti neuroni organizzati in strati, ogni strato trasforma un vettore in un altro vettore. Durante l'addestramento la rete regola i pesi da sola, e i vettori degli strati intermedi diventano delle **rappresentazioni** che nessuno ha progettato a mano.

I modelli linguistici sono reti di questo tipo, addestrate su enormi quantità di testo: il vettore che associano a ogni parola si chiama **embedding**.

## 2. La similarità del coseno

Dalla fisica: $\mathbf{a} \cdot \mathbf{b} = \|\mathbf{a}\|\,\|\mathbf{b}\| \cos\theta$. Quindi

$$\cos\theta = \frac{\mathbf{a} \cdot \mathbf{b}}{\|\mathbf{a}\|\,\|\mathbf{b}\|}$$

Vale 1 se i vettori puntano nella stessa direzione, 0 se sono perpendicolari, −1 se sono opposti. **Non dipende dalla lunghezza** dei vettori, solo dalla direzione. È la misura di somiglianza più usata per confrontare parole e frasi.

### ✏️ Esercizio 1

Completa la funzione `coseno` e verifica i tre casi.

In [ ]:
def coseno(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

a = np.array([1, 2])
print(coseno(a, np.array([3, 6])))     # stessa direzione   -> 1
print(coseno(a, np.array([-2, 1])))    # perpendicolari     -> 0
print(coseno(a, np.array([-1, -2])))   # opposti            -> -1

### Il coseno sui fiori

In [ ]:
print(specie[0], specie[1], specie[100])
print(coseno(X[0], X[1]))     # due setose
print(coseno(X[0], X[100]))   # setosa e virginica

Strano: anche setosa e virginica risultano molto simili (0,86). Le misure sono **tutte positive**, quindi tutti i vettori puntano nella stessa zona, lontano dall'origine. Spostiamo l'origine nel **centro** della nuvola di punti.

In [ ]:
Xc = X - X.mean(axis=0)       # dati centrati

print(coseno(Xc[0], Xc[1]))     # due setose
print(coseno(Xc[0], Xc[100]))   # setosa e virginica

Ora setosa e virginica puntano in direzioni quasi **opposte**.

### ✏️ Esercizio 2

Usa i dati centrati per trovare i 5 fiori più simili al fiore 0 secondo il coseno (i valori **più alti**). Di che specie sono?

In [ ]:
sim = np.array([coseno(Xc[0], Xc[i]) for i in range(len(Xc))])
simili = np.argsort(sim)[::-1][1:6]   # ordine decrescente, salto il fiore stesso
print(simili, specie[simili])

## 3. La matrice di similarità

Se rendiamo tutti i vettori lunghi 1, il coseno diventa un semplice prodotto scalare, e un prodotto tra matrici li calcola **tutti insieme**.

In [ ]:
Xn = Xc / np.linalg.norm(Xc, axis=1, keepdims=True)   # vettori di lunghezza 1
S = Xn @ Xn.T                                          # 150 x 150 coseni

plt.imshow(S, cmap="RdBu_r", vmin=-1, vmax=1)
plt.colorbar(label="similarità del coseno")
plt.title("Fiori in ordine: setosa, versicolor, virginica")
plt.show()

## 4. Le direzioni hanno un significato

La **differenza** tra due vettori è una freccia che va dall'uno all'altro: una **direzione**. Prendiamo la freccia che va dal fiore "medio" di setosa al fiore "medio" di virginica.

In [ ]:
media_setosa = X[specie == "setosa"].mean(axis=0)
media_virginica = X[specie == "virginica"].mean(axis=0)

direzione = media_virginica - media_setosa
direzione = direzione / np.linalg.norm(direzione)    # lunghezza 1
print(direzione)

Il prodotto scalare con un vettore di lunghezza 1 è la **proiezione**: dice quanto un punto è spostato lungo quella direzione. Proiettiamo tutti i fiori.

In [ ]:
proiezioni = Xc @ direzione

for nome in iris.target_names:
    plt.hist(proiezioni[specie == nome], bins=15, alpha=0.7, label=nome)
plt.xlabel("<-- verso setosa          verso virginica -->")
plt.legend()
plt.show()

Le tre specie si dispongono in ordine lungo la freccia, e versicolor sta in mezzo. Con le parole si può usare **esattamente questa tecnica**: si costruisce la direzione *uomo → donna* e si proiettano su di essa i nomi delle professioni. Che cosa vi aspettate?

## 5. Aritmetica tra vettori

Se le direzioni hanno un significato, possiamo **sommarle**. Costruiamo a mano un piccolo vocabolario con 4 caratteristiche: *regale*, *femminile*, *umano*, *animale*.

In [ ]:
#                       regale femminile umano animale
parole = {
    "re":      np.array([1, 0, 1, 0]),
    "regina":  np.array([1, 1, 1, 0]),
    "uomo":    np.array([0, 0, 1, 0]),
    "donna":   np.array([0, 1, 1, 0]),
    "gatto":   np.array([0, 0, 0, 1]),
    "gatta":   np.array([0, 1, 0, 1]),
    "corona":  np.array([1, 0, 0, 0]),
}

def piu_simile(vettore, escludi=()):
    candidati = [p for p in parole if p not in escludi]
    return max(candidati, key=lambda p: coseno(vettore, parole[p]))

v = parole["re"] - parole["uomo"] + parole["donna"]
print(v, "->", piu_simile(v, escludi=("re", "uomo", "donna")))

*re − uomo + donna*: partiamo da "re", togliamo "maschile umano", aggiungiamo "femminile umano". Otteniamo "regina".

### ✏️ Esercizio 3

1. Calcola *gatto − uomo + donna*. Che cosa ottieni?
2. Aggiungi al vocabolario "principe" e "principessa" con una quinta caratteristica *giovane* (ricordati di aggiungere uno 0 a tutte le altre parole!) e verifica *principe − uomo + donna*.

In [ ]:
v = parole["gatto"] - parole["uomo"] + parole["donna"]
print(v, "->", piu_simile(v, escludi=("gatto", "uomo", "donna")))   # gatta

#                         regale femminile umano animale giovane
parole = {
    "re":          np.array([1, 0, 1, 0, 0]),
    "regina":      np.array([1, 1, 1, 0, 0]),
    "uomo":        np.array([0, 0, 1, 0, 0]),
    "donna":       np.array([0, 1, 1, 0, 0]),
    "gatto":       np.array([0, 0, 0, 1, 0]),
    "gatta":       np.array([0, 1, 0, 1, 0]),
    "corona":      np.array([1, 0, 0, 0, 0]),
    "principe":    np.array([1, 0, 1, 0, 1]),
    "principessa": np.array([1, 1, 1, 0, 1]),
}
v = parole["principe"] - parole["uomo"] + parole["donna"]
print(v, "->", piu_simile(v, escludi=("principe", "uomo", "donna")))   # principessa

Qui le caratteristiche le abbiamo scelte noi, e i conti tornano perfettamente. Con caratteristiche **imparate** da un modello l'aritmetica funziona ancora, ma solo in modo approssimato, e talvolta con risultati sorprendenti.

## 6. Vedere 4 dimensioni in 2

La **PCA** cerca il piano su cui la nuvola di punti è più "spalmata" e proietta lì i dati. Perdiamo un po' di informazione, ma possiamo *vedere* lo spazio.

In [ ]:
pca = PCA(n_components=2)
coords = pca.fit_transform(X)

for nome in iris.target_names:
    plt.scatter(coords[specie == nome, 0], coords[specie == nome, 1], label=nome)
plt.xlabel("componente 1")
plt.ylabel("componente 2")
plt.legend()
plt.show()

print("Varianza conservata:", pca.explained_variance_ratio_)

### ✏️ Esercizio 4

Quanta parte dell'informazione (varianza) conservano le prime due componenti, in totale? Se proiettassimo in 2D delle parole che vivono in 100 dimensioni, la percentuale sarebbe più alta o più bassa? Perché questo dovrebbe renderci prudenti nel leggere quei grafici?

In [ ]:
print(f"Totale: {pca.explained_variance_ratio_.sum():.1%}")   # circa 98%
# Con 100 dimensioni la percentuale è molto più bassa: il grafico 2D è una proiezione
# parziale, e due punti vicini nel grafico possono essere lontani nello spazio originale.

## 🔗 Le nostre ipotesi

Come fa un modello linguistico a "sapere" che *gatto* e *cane* sono parole vicine, mentre *gatto* e *democrazia* no? Nessuno gli ha dato caratteristiche come "animale" o "domestico".

Scrivete qui le ipotesi della classe: le verificheremo più avanti.

- ...
- ...
- ...